# Coefficient Recovery Case Study: can `bunobee` recover known time-varying coefficients?

This notebook runs an end-to-end **coefficient-recovery experiment**:

1. A **pre-generated dataset** (`tvc-simple-2ch-level.nc`, simulated offline with `karpiu`) holds a marketing
   series whose data-generating process (DGP) is *fully known* — in particular, both the baseline
   **level** (a random walk) and the per-channel media coefficients (paths we chose) vary over time. The
   notebook only *reads* this file; `karpiu` is not needed to run it.
2. **`bunobee`** fits a structured **state-space model (SSP)** with a Kalman filter + RTS smoother and
   tries to recover the level and coefficient paths from the observed series alone.
3. A **disclosure** experiment then anchors the filter with the *true* coefficients at a few short windows
   (the "time-point prior" idea from `ssp_02`) and measures how much that improves recovery, inside and
   outside the windows.

## Why this is the cleanest possible test

The simulator that produced the dataset (`karpiu`'s `TimeVaryingCoefDailySimulator`)
is an **additive** daily generative model — deliberately *no* adstock, *no* saturation, *no* log-log
transform:

$$y_t \;=\; \text{level}_t \;+\; \sum_c \beta_{c,t}\,\frac{x_{c,t}}{\operatorname{median}(x_c)} \;+\; \varepsilon_t,
\qquad \varepsilon_t \sim \mathcal N(0,\sigma_h^2).$$

`bunobee`'s linear multi-series filter
[`kalman_filter_1d_st`](../../src/bunobee/models/ssp/kalman_1d_st.py) assumes exactly a linear-Gaussian
observation on top of a random-walk latent state:

$$a_t = a_{t-1} + \eta_t,\;\; \eta_t \sim \mathcal N(0,\operatorname{diag}(\sigma_q^2)),
\qquad y_{t} = Z_t\cdot a_t + \varepsilon_t.$$

The DGP's $\text{level}_t$ is itself a random walk
($\text{level}_t=\text{level}_{t-1}+\xi_t$, $\xi_t\sim\mathcal N(0,\sigma_\ell^2)$), which is exactly the
state transition `bunobee` assumes. So the two are **structurally matched**: set
$a_t=[\text{level}_t,\,\beta_{\text{search},t},\,\beta_{\text{social},t}]$
and $Z_t=[1,\,x_{\text{search},t}/\mathrm{med},\,x_{\text{social},t}/\mathrm{med}]$ and the `bunobee`
observation equation *is* the `karpiu` DGP. There is no model mismatch to blame — if `bunobee` cannot
recover the coefficients here, it cannot anywhere. This is the **optimistic benchmark**: a sanity
check on the estimator itself, isolated from the adstock/saturation nonlinearities a real MMM adds.

The level wanders as a random walk (`level_scale=2`, starting at 1000), so the intercept is *not* a
constant the estimator can take for granted. The two channels use two different coefficient shapes so
"time-varying" is unambiguous:

| state    | true path                                          | what it exercises |
|----------|----------------------------------------------------|-------------------|
| `level`  | random walk from 1000, innovation sd 2             | a stochastic drift |
| `search` | `linear` 50 → 150                                  | a monotone drift  |
| `social` | `sinusoid` base 80, amplitude 30, 1-year period    | an oscillation    |


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import numpyro
numpyro.set_host_device_count(4)
numpyro.enable_x64(True)  # level ~1e3 alongside betas ~1e2 — x64 keeps the filter well-conditioned
import jax
import jax.numpy as jnp
from jax import random
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.diagnostics import summary

# ---- estimator (bunobee) ----
from bunobee.models.ssp.kalman_1d_st import kalman_filter_1d_st   # linear multi-series filter
from bunobee.models.ssp.kalman_1d import kalman_rts_smoother_1d   # RTS smoother
from bunobee.models.ssp.plotting import plot_states
from bunobee.models.ssp.prior import disclosed_idx                 # disclosed timesteps from P_obs
from bunobee.simulation.ssp import construct_states_prior          # draws the disclosure windows

DATA_DIR = Path("../resource/tvc-simple-2ch-level")

In [ ]:
%load_ext watermark
%watermark -v -n -m -p numpy,pandas,xarray,matplotlib,jax,numpyro,bunobee -u -n

## 1. Load the ground-truth data

Two years of daily data (`n_steps=730`) so the `social` sinusoid completes two full cycles and the
`search` ramp is clearly visible. The dataset was simulated with `karpiu` using `level_scale=2.0` (a
random-walk intercept starting at 1000) and both seasonality flags off, so the time-varying level and media
coefficients are the *only* moving parts of the signal (apart from observation noise). It also stores the
ground truth — the full decomposition `gt_decomp` (`level`, per-channel contributions, `eps`), the coefficient
paths `gt_coef_ts` and the normalizer `gt_x_med` — and the simulation settings (`attrs`).

In [ ]:
ds = xr.open_dataset(DATA_DIR / "tvc-simple-2ch-level.nc")

regressors = [str(r) for r in ds["regressor"].values]          # ["search", "social"]
df = ds[["sales", *regressors]].to_dataframe().reset_index()    # observed: date, sales, search, social
coefs_ts = ds["gt_coef_ts"].to_pandas()                         # ground-truth betas: date x channel
level_ts = ds["gt_decomp"].sel(component="level").to_series()  # ground-truth level path (random walk)
x_med = ds["gt_x_med"].to_series()                              # per-channel median spend (the normalizer)

print("simulation settings:", {k: getattr(v, "item", lambda: v)() for k, v in ds.attrs.items()})
print("observed columns:", list(df.columns), "| shape:", df.shape)
print("median spend (normalizer):", {c: float(x_med[c]) for c in regressors})
df.head()

In [ ]:
# The observed series (top) and the ground-truth level / coefficient paths we will try to recover (bottom).
dates = df["date"].values
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
axes[0].plot(dates, df["sales"], color="black", lw=0.7)
axes[0].set_title("Observed response — sales (the only thing bunobee gets to see, besides spend)")
axes[0].set_ylabel("sales")
axes[1].plot(level_ts.index, level_ts.values, color="tab:purple", lw=1.5)
axes[1].set_title("Ground-truth level (random walk) — recovery target")
axes[1].set_ylabel("level")
for c in regressors:
    axes[2].plot(coefs_ts.index, coefs_ts[c], lw=1.8, label=f"true beta: {c}")
axes[2].set_title("Ground-truth time-varying coefficients — recovery target")
axes[2].set_ylabel("coefficient")
axes[2].set_xlabel("date")
axes[2].legend(fontsize=8)
fig.tight_layout()
plt.show()

## 2. Cast the additive model as a linear state space

Line the two models up term by term. The `karpiu` DGP is

$$y_t = \underbrace{\text{level}_t}_{\text{random walk}} + \beta_{\text{search},t}\,\frac{x_{\text{search},t}}{\mathrm{med}_{\text{search}}}
                                            + \beta_{\text{social},t}\,\frac{x_{\text{social},t}}{\mathrm{med}_{\text{social}}} + \varepsilon_t.$$

Define the **state** and the **design row**

$$a_t = \begin{bmatrix}\text{level}_t\\ \beta_{\text{search},t}\\ \beta_{\text{social},t}\end{bmatrix},
\qquad
Z_t = \Big[\,1,\;\; \tfrac{x_{\text{search},t}}{\mathrm{med}_{\text{search}}},\;\; \tfrac{x_{\text{social},t}}{\mathrm{med}_{\text{social}}}\,\Big],$$

and `bunobee`'s observation equation $y_t = Z_t\cdot a_t + \varepsilon_t$ reproduces the DGP exactly.
`bunobee`'s filter is written for the multi-series case (`Z` has shape `(n_steps, n_series, n_states)`);
here there is a single series, so `n_series = 1`.

Because we feed the response and design **on their natural scale** (no mean-normalization), the
recovered state path is directly comparable to `karpiu`'s true coefficients — no back-transform
needed.

In [ ]:
state_names = ["level", *regressors]
n_states = len(state_names)

xmed = np.array([float(x_med[c]) for c in regressors])        # (2,)
y = jnp.asarray(df["sales"].values[:, None])                   # (n_steps, 1)  single series
Zreg = df[regressors].values / xmed                            # (n_steps, 2)  spend / median
Z = jnp.asarray(np.concatenate([np.ones((len(df), 1)), Zreg], axis=1)[:, None, :])  # (n_steps, 1, 3)
n_steps, n_series, _ = Z.shape

print(f"y: {y.shape}   Z: {Z.shape}   (n_steps={n_steps}, n_series={n_series}, n_states={n_states})")
print("states:", state_names)

# sanity check: reconstruct sales at t=0 from the TRUE states and confirm it matches (up to noise)
true0 = np.array([level_ts.iloc[0], coefs_ts["search"].iloc[0], coefs_ts["social"].iloc[0]])
recon0 = float(np.asarray(Z)[0, 0] @ true0)
print(f"\nt=0  Z.a(true) = {recon0:.2f}   observed sales = {float(y[0,0]):.2f}   (diff = eps_0)")

## 3. Priors, and the one modeling assumption that matters

Two noise scales drive everything, and we put weakly-informative priors on both:

- **Observation noise** $\sigma_h \sim \text{HalfNormal}(20)$ — comfortably covers the true value of 5.
- **Process noise** $\sigma_q$, *one per state*. This is where we encode structure:

  $$\sigma_{q,\text{level}} \sim \text{HalfNormal}(5.0), \qquad
    \sigma_{q,\beta} \sim \text{HalfNormal}(2.0).$$

  Both priors are weakly informative and allow movement: the level is a genuine random walk (true
  $\sigma_{q,\text{level}}=2$), and the coefficients are allowed to drift too. The prior scales are the only
  assumption we hand the estimator — crucially, we do **not** tell it the *shape* of the coefficients
  (linear vs sinusoid), their amplitudes, or the level's path or noise scale. Everything else is learned.
  The hard part is **identifiability**: with a single series, a drifting level and drifting coefficients
  compete to explain the same low-frequency movement in sales.

The initial state `a0` starts the level at the data mean and the betas at **zero**, with a diffuse `P0`
— deliberately uninformative so recovery is driven by the data, not the prior.

In [ ]:
a0 = jnp.array([float(y.mean()), 0.0, 0.0])
P0 = jnp.diag(jnp.array([(0.3 * float(y.mean())) ** 2, 100.0 ** 2, 100.0 ** 2]))  # diffuse

sigma_h_scale = 20.0
sigma_q_scale = jnp.array([5.0, 2.0, 2.0])   # level: random walk | betas: time-varying

print("a0:", np.asarray(a0).round(2))
print("P0 diag:", np.diag(np.asarray(P0)).round(1))

## 4. Fit with NUTS

The NumPyro model samples the two noise scales, runs the linear Kalman filter to get the Gaussian
log-likelihood (`logp=True`) and the filtered state path `at`, then runs the RTS smoother to get the
smoothed path `at_smooth`. `numpyro.factor` injects the filter's likelihood; the state paths and the
fitted mean `mu` are recorded as deterministics so every posterior draw carries its own trajectory.

In [ ]:
def model(a_obs=None, P_obs=None):
    """NumPyro model; ``a_obs`` / ``P_obs`` (n_steps, n_states) optionally disclose latent states."""
    sigma_h = numpyro.sample("sigma_h", dist.HalfNormal(sigma_h_scale * jnp.ones(n_series)))
    sigma_q = numpyro.sample("sigma_q", dist.HalfNormal(sigma_q_scale))

    lp, at, Pt, *_ = kalman_filter_1d_st(
        a0=a0, P0=P0, Z=Z, sigma_h=sigma_h, sigma_q=sigma_q, y=y, logp=True, a_obs=a_obs, P_obs=P_obs,
    )
    numpyro.factor("lp", lp)
    numpyro.deterministic("at", at)
    numpyro.deterministic("mu", jnp.sum(Z * at[:, None, :], axis=-1))  # fitted mean (n_steps, n_series)

    # Marginal state variances: the filter's own covariance is the *dominant* source of state
    # uncertainty (hyperparameter spread across draws is secondary), so we keep it and the
    # smoother's P_{t|T} to build calibrated bands in section 5.
    Pt_diag = jnp.diagonal(Pt, axis1=1, axis2=2)                       # filtered marginal variance
    at_smooth, Ps_diag = kalman_rts_smoother_1d(at, Pt_diag, sigma_q)  # smoothed mean + variance
    numpyro.deterministic("at_var", Pt_diag)
    numpyro.deterministic("at_smooth", at_smooth)
    numpyro.deterministic("at_smooth_var", Ps_diag)


# quick single-pass sanity check before paying for MCMC
lp0, at0, *_ = kalman_filter_1d_st(
    a0=a0, P0=P0, Z=Z, sigma_h=jnp.array([10.0]), sigma_q=sigma_q_scale, y=y, logp=True,
)
print(f"test filter pass -> logp = {float(lp0):.1f}, at shape = {at0.shape}")

In [ ]:
def fit(model, *args, seed=0, num_warmup=500, num_samples=500):
    """Run NUTS on ``model(*args)`` and return the finished MCMC object."""
    mcmc = MCMC(NUTS(model), num_warmup=num_warmup, num_samples=num_samples, num_chains=4, progress_bar=False)
    mcmc.run(random.PRNGKey(seed), *args)
    jax.block_until_ready(mcmc.get_samples())  # JAX dispatches asynchronously; wait so timings are honest
    return mcmc


t0 = time.time()
mcmc = fit(model)  # baseline: no disclosure
print(f"MCMC wall time: {time.time() - t0:.1f}s")

In [ ]:
numpyro_post = mcmc.get_samples(group_by_chain=True)
mcmc.print_summary(exclude_deterministic=True)

# convergence flags on the scalar params
diag = summary(numpyro_post, prob=0.9)
print("--- convergence flags (r_hat > 1.01 or n_eff < 100) ---")
flagged = False
for p, s in diag.items():
    rhat = float(np.asarray(s["r_hat"]).max()); neff = float(np.asarray(s["n_eff"]).min())
    if rhat > 1.01 or neff < 100:
        print(f"  WARN {p:<12} r_hat={rhat:.3f} n_eff={neff:.0f}"); flagged = True
if not flagged:
    print("  none — all scalar sites converged")

print("\nRecall the truth:  sigma_h = 5.0  |  sigma_q[level] = 2.0  (random-walk innovation sd)")

## 5. Did it recover?

The headline metric. For each state we compare the **smoothed** posterior path against the true path on
three axes:

- **MAE** — mean absolute error between the posterior-median path and the truth.
- **corr** — correlation between recovered and true paths.
- **90% coverage** — the fraction of true points that fall inside the pointwise 90% credible band. A
  well-calibrated recovery lands near 0.90.

**Building an honest band.** The posterior over each latent path has *two* sources of spread: the
Kalman smoother's own state covariance $P_{t|T}$ for a *given* $(\sigma_h,\sigma_q)$, and the variation
of the smoothed mean *across* posterior draws of those hyperparameters. The first dominates here (730
points pin the noise scales tightly), and ignoring it collapses the band. We combine both by drawing one
state sample per posterior draw, $a_t^{(s)} \sim \mathcal N\big(\hat a^{(s)}_{t|T},\, P^{(s)}_{t|T}\big)$,
and reading quantiles off the pooled draws — the standard way to propagate state *and* hyperparameter
uncertainty together.

In [ ]:
post = mcmc.get_samples()

true = np.column_stack([
    level_ts.values,
    coefs_ts["search"].values,
    coefs_ts["social"].values,
])


def pooled_state_draws(post, mean_key, var_key, rng):
    """One draw from N(mean, var) per posterior sample -> (n_samples, n_steps, n_states)."""
    m = np.asarray(post[mean_key])
    sd = np.sqrt(np.clip(np.asarray(post[var_key]), 0.0, None))
    return m + sd * rng.standard_normal(m.shape)


def recovered_states(post, seed=1):
    """Pooled filtered / smoothed state draws, plus the smoothed median and 90% band."""
    rng = np.random.default_rng(seed)
    at_d = pooled_state_draws(post, "at", "at_var", rng)               # filtered (full uncertainty)
    sm_d = pooled_state_draws(post, "at_smooth", "at_smooth_var", rng)  # smoothed (full uncertainty)
    lo, hi = np.quantile(sm_d, [0.05, 0.95], axis=0)
    return at_d, sm_d, np.median(sm_d, axis=0), lo, hi


def score_recovery(med, lo, hi, mask=None):
    """MAE, correlation, 90% coverage and mean band width per state, optionally on a time mask."""
    mask = np.ones(n_steps, dtype=bool) if mask is None else mask
    rows = []
    for i, nm in enumerate(state_names):
        t, m = true[mask, i], med[mask, i]
        corr = float(np.corrcoef(m, t)[0, 1]) if t.std() > 1e-9 else np.nan
        rows.append({
            "state": nm,
            "MAE": float(np.mean(np.abs(m - t))),
            "corr": corr,
            "90% coverage": float(np.mean((t >= lo[mask, i]) & (t <= hi[mask, i]))),
            "90% band width": float(np.mean(hi[mask, i] - lo[mask, i])),
        })
    return pd.DataFrame(rows).set_index("state")


at_draws, at_smooth_draws, sm_med, sm_lo, sm_hi = recovered_states(post)
recovery = score_recovery(sm_med, sm_lo, sm_hi)
recovery.round(4)

In [ ]:
# Headline recovery plot: true time-varying coefficient vs smoothed posterior (median + 90% band).
fig, axes = plt.subplots(1, n_states, figsize=(16, 4.2))
for i, (ax, nm) in enumerate(zip(axes, state_names)):
    ax.fill_between(dates, sm_lo[:, i], sm_hi[:, i], color="darkgreen", alpha=0.22, label="90% CI (smoothed)")
    ax.plot(dates, sm_med[:, i], color="darkgreen", lw=1.2, label="recovered (median)")
    ax.plot(dates, true[:, i], color="black", lw=1.6, ls="--", label="ground truth")
    ax.set_title(f"{nm}", fontsize=11)
    ax.tick_params(axis="x", labelsize=7, rotation=30)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=8, loc="best")
fig.suptitle("Coefficient recovery: bunobee smoothed posterior vs karpiu ground truth", y=1.02)
fig.tight_layout()
plt.show()

`bunobee`'s own `plot_states` gives the filtered-vs-smoothed view straight from the posterior — the
smoothed band is tighter because it conditions on the whole series, not just the past. (No ground-truth
overlay here: `plot_states`' `coefs_df` argument draws a *constant* reference line, which would be
misleading for genuinely time-varying coefficients — the dashed truth lines above are the right view.)

In [ ]:
fig, _ = plot_states(
    {"at": at_draws, "at_smooth": at_smooth_draws},
    dates=dates,
    state_labels=state_names,
    states_key=["at", "at_smooth"],
    colors={"at": "steelblue", "at_smooth": "darkgreen"},
    n_cols=3,
    title="Filtered (at) vs smoothed (at_smooth) latent states — posterior median + 90% CI",
)
plt.show()

## 6. Disclosure: anchoring the filter with known coefficients

Notebook `ssp_02` introduced **time-point priors**: at a few timesteps we tell the filter what some latent
states are, with an uncertainty, and the filter fuses that information into its state estimate (a
precision-weighted update before each step). In the real world those anchors would come from outside
knowledge — a lift test, an experiment, a trusted earlier estimate. Here we are in a simulation, so we can
*disclose the ground truth* and measure exactly how much an anchor helps.

The setup:

- **What is disclosed:** the two media coefficients $\beta_{\text{search},t}$ and $\beta_{\text{social},t}$ at
  `N_PERIODS` randomly placed windows of `N_POINTS` consecutive days, observed with noise sd
  `DISC_OBS_SIGMA`. The **level is never disclosed** (its variance stays infinite), so any improvement in the
  level is indirect.
- **Time-varying truth:** `construct_states_prior` takes one `true_states` vector, which is right for the
  constant coefficients in `ssp_02` but wrong here. We use it only to draw the windows and then write the
  *true value at each disclosed day* (`gt_coef_ts[t]`) into `a_obs`.
- **Fair comparison:** the model, priors, data and MCMC seed are identical to section 4. The only difference
  is the disclosure passed to the filter, so any change in recovery is due to it.

Recovery is scored separately **inside** the disclosed windows (where the filter is told the answer, so
good scores are close to guaranteed) and **outside** them (where the question is whether the anchors
propagate to the undisclosed days).

In [ ]:
# ---- disclosure settings (simulation-only: the disclosed values are the ground truth) ----
N_PERIODS = 3          # number of disclosure windows
N_POINTS = 10          # consecutive days per window
DISC_OBS_SIGMA = 2.0   # sd of the disclosed coefficients, in coefficient units (true betas are ~50-150)
DISC_SEED = 42         # which windows are drawn


def build_disclosure(seed=DISC_SEED, n_periods=N_PERIODS, n_points=N_POINTS, obs_scale=DISC_OBS_SIGMA):
    """Disclose the true time-varying betas at randomly drawn windows.

    Parameters
    ----------
    seed : int
        RNG seed for the window positions.
    n_periods, n_points : int
        Number of windows and consecutive steps per window.
    obs_scale : float
        Standard deviation of the disclosed coefficients.

    Returns
    -------
    states_prior : xr.Dataset
        ``a_obs`` / ``P_obs`` / ``positivity`` with dims ``(time, state)``; ``P_obs`` is ``inf`` wherever
        nothing is disclosed (including the whole level column).
    idx : np.ndarray
        Time indices of the disclosed steps.
    """
    # true[0] is a placeholder: construct_states_prior only uses it to fill a_obs, which is overwritten below
    sp = construct_states_prior(
        n_steps=n_steps, n_states=n_states, true_states=jnp.asarray(true[0]), regressors=regressors,
        n_periods=n_periods, n_points=n_points, seed=seed, obs_scale=obs_scale,
    )
    idx = disclosed_idx(sp)
    a_obs = np.zeros((n_steps, n_states))
    a_obs[idx, 1:] = true[idx, 1:]            # the true beta *at each disclosed day*; level column unused
    return sp.assign(a_obs=(("time", "state"), a_obs)), idx


states_prior, win_idx = build_disclosure()
a_obs = jnp.asarray(states_prior["a_obs"].values)
P_obs = jnp.asarray(states_prior["P_obs"].values)
in_win = np.zeros(n_steps, dtype=bool)
in_win[win_idx] = True

print(f"disclosed {in_win.sum()} of {n_steps} days ({in_win.mean():.1%}) in {N_PERIODS} windows")
print("disclosed day indices:", win_idx)
print("a_obs at first disclosed day:", np.asarray(a_obs[win_idx[0]]).round(2),
      "| true:", true[win_idx[0]].round(2))

In [ ]:
t0 = time.time()
mcmc_d = fit(model, a_obs, P_obs)  # same seed and settings as the baseline, plus the disclosure
print(f"MCMC wall time: {time.time() - t0:.1f}s")
post_d = mcmc_d.get_samples()
_, sm_d_draws, med_d, lo_d, hi_d = recovered_states(post_d)

arms = {"no disclosure": (sm_med, sm_lo, sm_hi), "disclosure": (med_d, lo_d, hi_d)}
regions = {"outside windows": ~in_win, "inside windows": in_win}
comparison = pd.concat(
    {(region, arm): score_recovery(*stats, mask=mask)
     for region, mask in regions.items() for arm, stats in arms.items()},
    names=["region", "arm"],
).reorder_levels(["state", "region", "arm"]).sort_index()
comparison.round(3)

In [ ]:
# True path vs the two recovered bands; shaded strips mark the disclosed windows.
fig, axes = plt.subplots(1, n_states, figsize=(16, 4.2))
for i, (ax, nm) in enumerate(zip(axes, state_names)):
    for k in np.flatnonzero(np.diff(np.r_[0, in_win.astype(int), 0]) == 1):   # window starts
        stop = k + np.argmax(~in_win[k:]) if (~in_win[k:]).any() else n_steps
        ax.axvspan(dates[k], dates[stop - 1], color="gold", alpha=0.35, lw=0)
    ax.fill_between(dates, sm_lo[:, i], sm_hi[:, i], color="steelblue", alpha=0.25, label="no disclosure 90% CI")
    ax.fill_between(dates, lo_d[:, i], hi_d[:, i], color="darkorange", alpha=0.30, label="disclosure 90% CI")
    ax.plot(dates, sm_med[:, i], color="steelblue", lw=1.0)
    ax.plot(dates, med_d[:, i], color="darkorange", lw=1.0)
    ax.plot(dates, true[:, i], color="black", lw=1.5, ls="--", label="ground truth")
    ax.set_title(nm, fontsize=11)
    ax.tick_params(axis="x", labelsize=7, rotation=30)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=7, loc="best")
fig.suptitle("Coefficient recovery with vs without disclosure (gold = disclosed windows)", y=1.02)
fig.tight_layout()
plt.show()

### Final comparison: do time-point priors help on the days we did *not* disclose?

The windows are random, so a single comparison can be lucky or unlucky. We repeat the experiment over
several window draws and over the number of windows, refit each time (with shorter 300+300 chains), and score **only the
undisclosed
days** — where the anchors must propagate through the state dynamics to help. The no-disclosure fit is the
same in every row; it is scored on each draw's undisclosed days so both arms see identical days.

The table reports, per state and number of windows (mean over window draws):

- **MAE** with and without time-point priors, and the relative change (negative = better).
- **90% coverage** of the true path with and without (target ≈ 0.90).
- **Band width** — the mean width of the 90% band, to check that any coverage gain is not just a wider band.

In [ ]:
N_SEEDS = 6
WINDOW_GRID = [1, 3, 6]   # number of disclosure windows

rows = []
t0 = time.time()
for nw in WINDOW_GRID:
    for s in range(N_SEEDS):
        sp, idx = build_disclosure(seed=s, n_periods=nw)
        mask = np.ones(n_steps, dtype=bool)
        mask[idx] = False                                              # undisclosed days only
        post_s = fit(model, jnp.asarray(sp["a_obs"].values), jnp.asarray(sp["P_obs"].values),
                     num_warmup=300, num_samples=300).get_samples()   # shorter chains: many refits
        _, _, m_s, lo_s, hi_s = recovered_states(post_s)
        for arm, stats in {"without": (sm_med, sm_lo, sm_hi), "with": (m_s, lo_s, hi_s)}.items():
            tab = score_recovery(*stats, mask=mask)
            for nm in state_names:
                rows.append({"windows": nw, "seed": s, "arm": arm, "state": nm, "MAE": tab.loc[nm, "MAE"],
                             "coverage": tab.loc[nm, "90% coverage"], "width": tab.loc[nm, "90% band width"]})
print(f"{len(WINDOW_GRID) * N_SEEDS} fits in {time.time() - t0:.0f}s")
sweep = pd.DataFrame(rows)

In [ ]:
# Final table: undisclosed days only, mean over window draws; "with" = time-point priors (disclosure).
avg = sweep.groupby(["state", "windows", "arm"])[["MAE", "coverage", "width"]].mean().unstack("arm")
final = pd.DataFrame({
    ("MAE", "without"): avg[("MAE", "without")],
    ("MAE", "with"): avg[("MAE", "with")],
    ("MAE", "change %"): 100 * (avg[("MAE", "with")] / avg[("MAE", "without")] - 1),
    ("90% coverage", "without"): avg[("coverage", "without")],
    ("90% coverage", "with"): avg[("coverage", "with")],
    ("90% band width", "without"): avg[("width", "without")],
    ("90% band width", "with"): avg[("width", "with")],
})
final.columns = pd.MultiIndex.from_tuples(final.columns)
final.round(3)

## Takeaways

- **The coefficients are recovered well without any anchor.** On the matched, additive DGP `bunobee`'s linear
  state-space model tracks both time-varying coefficient paths (corr ≈ 0.996 for `search`, ≈ 0.98 for
  `social`; 90% coverage ≈ 0.91) without being told their shapes or amplitudes, even though the level is
  moving at the same time.
- **The level is recovered more coarsely.** It follows the slow trend (corr ≈ 0.92) but smooths away the
  short-lived wiggles, and its 90% band covers only ≈ 75% of the true path. The posterior of
  $\sigma_{q,\text{level}}$ lands near 1.2 against a true 2.0, and the band is too narrow for the same reason:
  with one series, a drifting level and drifting coefficients compete for the same low-frequency movement.
- **Time-point priors improve the *undisclosed* days, and the gain grows with the number of windows.**
  Scoring only the days we did not disclose, the MAE of the coefficients falls by ≈ 5% (`search`) and ≈ 1%
  (`social`) with 1 window of 10 days, ≈ 18% and ≈ 5% with 3 windows, and ≈ 28% and ≈ 11% with 6 windows. The
  anchors propagate through the random-walk dynamics, so the benefit is real but local, and it is larger for
  the monotone `search` ramp than for the oscillating `social` path, which drifts away from an anchor faster.
- **Coverage is not hurt, and the bands tighten.** Coverage of the undisclosed days stays at or slightly above
  the no-anchor value for the coefficients (e.g. `search`: 0.91 → 0.94 with 6 windows) while the 90% band gets
  narrower, so the gain is not bought with wider intervals.
- **The level gets a smaller, indirect benefit and its coverage does not improve.** The level is never
  disclosed; pinning the betas in a window pins `sales − Z·β` there, so its MAE falls ≈ 10% with 6 windows,
  but its coverage stays ≈ 0.73–0.75 (slightly lower, because the band narrows while the error is dominated by
  the unmodelled fast variation). Fixing the coefficients does not fix the level's under-dispersion.
- **This benchmark is optimistic on the disclosure side too.** The disclosed values are the exact truth plus
  small noise. A biased anchor would pull the fit the wrong way with a *narrower* band, so coverage could
  degrade rather than improve; that case is not tested here.
- **This is still the optimistic benchmark, by design.** There is no adstock, no saturation, no seasonality
  and no model mismatch here — so this measures the *estimator*, not an MMM. A real campaign adds carryover
  and diminishing returns on top; recovery under those nonlinearities is a strictly harder, separate
  question that this clean baseline is the right starting point for.